In [1]:
import os
import pandas as pd
import numpy as np
import netCDF4
import matplotlib.pyplot as plt
import numpy as np

from polar2uv import polar2uv
from uv2polar import uv2polar
from DataFrame_maker import *

def getNombreBoya(ruta_a_archivo):
    ruta_normalizada = os.path.normpath(ruta_a_archivo)
    partes = ruta_normalizada.split(os.path.sep)
    nombre_boya = partes[-3].replace("-", "_")
    return nombre_boya

def selector_de_archivos(ruta, mensaje, tipo):
    """"Devuelve la ruta al archivo crudo y al archivo validado"""
    archivos = os.listdir(ruta)
    ruta_de_archivos = []
    for archivo in archivos:  
        if f"msj{mensaje}" in archivo and f"_validados_{tipo}_" in archivo and archivo.endswith('.pkl'):
            ruta_de_archivos.append(os.path.normpath(os.path.join(ruta, archivo)))
        
    return ruta_de_archivos

def datenum_to_datetime(datenum):
    import datetime
    """
    Descripción:
        Convierte un array en formato datenum de MATLAB a objetos datetime de Python, redondeando al minuto más cercano y manejando MaskedArray.

    Parámetros:
        datenum (array-like): Array de números datenum (puede ser MaskedArray).

    Retorna:
        np.ndarray: Array de objetos datetime convertidos y redondeados al minuto.

    Ejemplo:
        >>> datenum_to_datetime([738000.501, 738001.499])

    Categoría:
        Funciones-Comunes

    Funciones auxiliares:
        np.array, datetime.datetime.fromordinal, datetime.timedelta
    """
    # Primero convierte el maskedArray en un array normal de python
    if isinstance(datenum, np.ma.MaskedArray):
        fechas = np.array(datenum.filled(np.nan))
    else:
        fechas = np.array(datenum)

    # Lista para almacenar las fechas convertidas
    fechas_convertidas = []
    for idatenum in fechas.flatten():
        # Conversion de idatenum a un valor escalar
        idatenum = float(idatenum)
        # Convierte un número datenum de MATLAB a datetime de Python
        fecha = datetime.datetime.fromordinal(int(idatenum)) + datetime.timedelta(days=idatenum % 1) - datetime.timedelta(days=366)
        # Redondear al minuto más cercano
        fecha = fecha.replace(microsecond=0)
        if fecha.second >= 30:
            fecha += datetime.timedelta(minutes=1)
        fecha = fecha.replace(second=0)
        fechas_convertidas.append(fecha)
    # Retorna las fechas convertidas como un array de numpy
    return np.array(fechas_convertidas)

def verificar_arrays_iguales(array1, array2):
    """
    Verifica si dos arrays son iguales.

    Parámetros:
        array1 (array-like): Primer array a comparar.
        array2 (array-like): Segundo array a comparar.

    Retorna:
        bool: True si los arrays son iguales, False en caso contrario.
    """
    return np.array_equal(array1, array2)

In [2]:
# Configuraciones [Acá se indica que carpetas, mensaje y tipo de mensaje se cargará] 
carpeta = "//192.168.15.249/Med_2025-2026/Reportes_Edit/Reporte_1.3/2026/03. Marzo/BOT1-02-T80/DATOS"
nombre_del_archivo = "msj4_validados_mem_2025111619_2026020821.pkl"
ruta_de_archivo = os.path.normpath(os.path.join(carpeta, nombre_del_archivo))    
pkl = leer_datos_de_pickle(ruta_de_archivo)

In [3]:
# Cargar NC
carpeta_al_nc = "//192.168.15.249/Med_2025-2026/Reportes_Edit/Reporte_1.3/2026/03. Marzo/BOT1-02-T80/RECUPERACIONES/BOT1-02-T80/CORRIENTES/VALIDADOS"
carpeta_al_nc = os.path.normpath(carpeta_al_nc)
ruta_al_nc = os.path.join(carpeta_al_nc, "BOT1-02-T80-SIG500DW-NS103971-Z0.54-MEM-20251116-20260208.nc")
nc = netCDF4.Dataset(ruta_al_nc, 'r')

In [4]:
print(f"NC = {nc.variables.keys()}")
print(f"PKL = {pkl.columns}")

NC = dict_keys(['nam', 'Lat', 'Lon', 'TiranteDiseno', 'ProfDiseno', 'TiranteEstimado', 'depth', 'jd', 'Temp', 'u', 'v', 'Heading', 'Pitch', 'Roll', 'ae'])
PKL = Index(['Battery', 'Heading', 'Pitch', 'Roll', 'Pressure', 'Temp', 'StdHeading',
       'StdPitch', 'StdRoll', 'StdPressure', 'u', 'v', 'Up1', 'Up2',
       'PercentageGood', 'Niveles', 'Rap', 'Dir', 'Lat', 'Lon',
       'TiranteDiseno', 'TiranteEstimado', 'nombreBoya', 'Tab',
       'origenDeLosDatos', 'tipoDeEntregable', 'NoDatalogger',
       'ProfDisenoCorr', 'ModeloInstCorr', 'NoSerieInstCorr', 'ProfIni'],
      dtype='object')


In [5]:
var_nc = "rap"
var_pkl = "rap"

In [17]:
rap_pkl = np.full((2019, 12), np.nan)  # Ajusta el tamaño según tus necesidades
dir_pkl = np.full((2019, 12), np.nan)  # Ajusta el tamaño según tus necesidades
# Quitar NaNs del vector
for iday in range(0,len(pkl["Rap"])):
    rap_tmp = np.array(pkl["Rap"][iday])
    rap_clean = rap_tmp[~np.isnan(rap_tmp)]
    
    dir_tmp = np.array(pkl["Dir"][iday])
    dir_clean = dir_tmp[~np.isnan(dir_tmp)]
    
    # Asignar como columna en u (col_idx = índice de columna destino)
    rap_pkl[iday, 0:len(rap_clean)] = rap_clean
    dir_pkl[iday, 0:len(dir_clean)] = dir_clean
    
u_pkl, v_pkl = polar2uv(dir_pkl, rap_pkl)

C:\Users\Atmosfera\AppData\Local\Temp\ipykernel_1700\261858357.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  rap_tmp = np.array(pkl["Rap"][iday])
C:\Users\Atmosfera\AppData\Local\Temp\ipykernel_1700\261858357.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dir_tmp = np.array(pkl["Dir"][iday])


In [18]:
u_nc = np.array(nc.variables["u"][:]).astype(np.float32)
v_nc = np.array(nc.variables["v"][:]).astype(np.float32)
u_pkl = -1*np.array(u_pkl).astype(np.float32)
v_pkl = -1*np.array(v_pkl).astype(np.float32)

In [22]:
dir_nc, rap_nc = uv2polar(u_nc, v_nc)
for iday in range(0, 2):
    print(f"u = {u_pkl[iday,0]} u_nc = {u_nc[iday,0]}")
    print(f"v = {v_pkl[iday,0]} v_nc = {v_nc[iday,0]}")
    
    
# print(f"Resultados para u: {u_result}")
# print(f"Resultados para v: {v_result}")

u = 0.0284000001847744 u_nc = 0.0284000001847744
v = -0.1688999980688095 v_nc = -0.1688999980688095
u = -0.09700000286102295 u_nc = -0.09700000286102295
v = -0.23389999568462372 v_nc = -0.23389999568462372


In [24]:
rap_pkl[0,:]
dir_pkl[0,:]

array([350.45518249,   4.34725101,  29.61769887,  37.13993914,
        46.4149221 ,  98.74616226, 227.67734862, 224.48691718,
        69.44395478,  36.89275196,  11.72893165,          nan])

In [2]:
dir, rap = uv2polar(0.0284, -0.1688)

In [3]:
dir

np.float64(170.44963209944498)